In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc gem-suite matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Nishimori phase transition

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Tantiya ng paggamit: 3 minuto sa isang Heron r2 processor (PAALALA: Ito ay tantiya lamang. Maaaring mag-iba ang iyong runtime.)*
## Mga layunin ng pag-aaral
Pagkatapos ng tutorial na ito, dapat inaasahan ng mga gumagamit ang mga sumusunod na kinalabasan:
- Maunawaan ang Nishimori phase transition at kung paano ito lumalabas bilang pagpapakita ng long-range entanglement sa random-bond Ising model.
- Ipatupad ang *generation of entanglement by measurement* (GEM) protocol sa quantum hardware gamit ang mga mid-circuit measurement at constant-depth circuit.
- Ikarakterisa ang transisyon sa pamamagitan ng pagkuha ng two-point correlation at normalized variance ng magnetization mula sa experimental data.

## Mga kinakailangan
Inirerekomenda namin ang pagiging pamilyar sa mga sumusunod na paksa bago dumaan sa tutorial na ito:
- Ang gabay na [Measure qubits](/guides/measure-qubits), partikular ang seksyon sa mid-circuit measurement na inaaasahang gamitin ng GEM protocol.
- [Exact and noisy simulation with Qiskit Aer primitives](/guides/simulate-with-qiskit-aer), kung paano isinasagawa ang small-scale na seksyon.
- [Long-range entanglement with dynamic circuits](/tutorials/long-range-entanglement), isang kasamang tutorial na gumagamit ng parehong measurement-based-entanglement na paradigma.
- [Heavy-hex lattice](https://www.ibm.com/quantum/blog/heavy-hex-lattice), ang IBM&reg; hardware topology kung saan itinayo ang plaquette lattice.
## Background
Ipinapakita ng tutorial na ito kung paano irealisasyon ang Nishimori phase transition sa isang quantum processor. Ang eksperimentong ito ay orihinal na inilarawan sa [*Realizing the Nishimori transition across the error threshold for constant-depth quantum circuits*](https://arxiv.org/abs/2309.02863).

Ang Nishimori phase transition ay tumutukoy sa transisyon sa pagitan ng short-range at long-range ordered phases sa random-bond Ising model. Sa isang quantum computer, ang long-range ordered phase ay lumalabas bilang isang estado kung saan ang mga qubit ay nag-entangle sa buong device. Ang lubhang naka-entangle na estadong ito ay inihahanda gamit ang *generation of entanglement by measurement* (GEM) protocol. Sa pamamagitan ng paggamit ng mid-circuit measurements, ang GEM protocol ay nakakapag-entangle ng mga qubit sa buong device gamit ang mga circuit na may constant depth lamang. Ginagamit ng tutorial na ito ang implementasyon ng GEM protocol mula sa [GEM Suite](https://github.com/qiskit-community/gem-suite) software package.
## Mga kinakailangan sa pag-install
Bago magsimula sa tutorial na ito, siguraduhing mayroon kang sumusunod na naka-install:

- Qiskit SDK v1.0 o mas bago, na may suporta sa [visualization](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 o mas bago (`pip install qiskit-ibm-runtime`)
- Qiskit Aer v0.14 o mas bago (`pip install qiskit-aer`)
- GEM Suite (`pip install gem-suite`)
## Setup

In [1]:
import matplotlib.pyplot as plt
import warnings

from collections import defaultdict

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_aer import AerSimulator

from qiskit.transpiler import generate_preset_pass_manager

from gem_suite import PlaquetteLattice
from gem_suite.experiments import GemExperiment

## Maliit na halimbawa sa simulator

Sa seksyong ito, ang buong workflow ay isinasagawa sa walang ingay na `AerSimulator`. Ang plaquette lattice ay pinaghigpit sa isang plaquette lamang (12 qubit) upang ang simulation ay manatiling maliit at mabilis, habang naisasagawa pa rin ang bawat bahagi ng GEM protocol: mid-circuit measurement, ang sweep ng $R_{ZZ}$ angle, decoding, at ang normalized-variance analysis. Ang parehong workflow ay palalawakin sa maraming plaquette at sa buong lattice sa tunay na hardware sa ibang pagkakataon.

### Hakbang 1: I-map ang mga classical input sa isang quantum problem

Ang GEM protocol ay gumagana sa isang quantum processor na may qubit connectivity na inilarawan ng isang lattice. Ang mga IBM Quantum&reg; processor ngayon ay gumagamit ng [heavy-hex lattice](https://www.ibm.com/quantum/blog/heavy-hex-lattice). Ang mga qubit ng processor ay pinagsasama-sama sa mga *plaquette* batay sa kung aling unit cell ng lattice ang kanilang inookupahan. Dahil maaaring lumitaw ang isang qubit sa higit sa isang unit cell, ang mga plaquette ay hindi disjoint. Sa heavy-hex lattice, ang isang plaquette ay naglalaman ng 12 qubit. Ang mga plaquette mismo ay bumubuo rin ng lattice, kung saan dalawang plaquette ay konektado kung may mga qubit silang pinagsasaluhan. Sa heavy-hex lattice, ang mga kalapit na plaquette ay nagbabahagi ng tatlong qubit.

Sa GEM Suite software package, ang pangunahing class para sa pagpapatupad ng GEM protocol ay ang `PlaquetteLattice`, na kumakatawan sa lattice ng mga plaquette (na iba sa heavy-hex lattice). Ang isang `PlaquetteLattice` ay maaaring i-initialize mula sa isang qubit coupling map. Sa kasalukuyan, tanging ang heavy-hex coupling maps lamang ang sinusuportahan.

Ang sumusunod na code cell ay nag-i-initialize ng plaquette lattice mula sa coupling map ng isang quantum processing unit (QPU). Ang plaquette lattice ay hindi palaging sumasaklaw sa buong hardware. Halimbawa, ang `ibm_torino` ay may 133 kabuuang qubit, ngunit ang pinakamalaking plaquette lattice na umaangkop sa device ay gumagamit lamang ng 125 sa kanila, na binubuo ng 18 plaquette; ang `ibm_pittsburgh` (156 qubit) ay katulad na nag-aakma ng 144 qubit sa 21 plaquette. Ang parehong pattern ay nananatili para sa iba pang heavy-hex QPU na may iba't ibang bilang ng qubit.

In [ ]:
# QiskitRuntimeService.save_account(channel="ibm_quantum", token="<YOUR_API_KEY>", overwrite=True,
# set_as_default=True)
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
aer_backend = AerSimulator.from_backend(backend)
plaquette_lattice = PlaquetteLattice.from_coupling_map(backend.coupling_map)

print(f"Number of qubits in backend: {backend.num_qubits}")
print(
    f"Number of qubits in plaquette lattice: {len(list(plaquette_lattice.qubits()))}"
)
print(f"Number of plaquettes: {len(list(plaquette_lattice.plaquettes()))}")

Maaari mong ivisualize ang plaquette lattice sa pamamagitan ng paggawa ng diagram ng graph representation nito. Sa diagram, ang mga plaquette ay kinakatawan ng mga naka-label na hexagon, at dalawang plaquette ay konektado ng isang edge kung may mga qubit silang pinagsasaluhan.

In [3]:
plaquette_lattice.draw_plaquettes()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/625882a4-faeb-4d96-b441-c989f43c4dea-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/625882a4-faeb-4d96-b441-c989f43c4dea-0.avif)

Maaari mong kunin ang impormasyon tungkol sa mga indibidwal na plaquette, tulad ng mga qubit na nilalaman nila, gamit ang `plaquettes` method.

In [4]:
# Get a list of the plaquettes
plaquettes = list(plaquette_lattice.plaquettes())
# Display information about plaquette 0
plaquettes[0]

PyPlaquette(index=0, qubits=[3, 4, 5, 6, 7, 16, 17, 23, 24, 25, 26, 27], neighbors=[4, 3, 1])

You can also produce a diagram of the underlying qubits that form the plaquette lattice.

In [5]:
plaquette_lattice.draw_qubits()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/a19d63ce-3572-4081-a008-c1332fbbe303-0.avif" alt="Output of the previous code cell" />

Maaari ka ring gumawa ng diagram ng mga pinagbabatayan na qubit na bumubuo sa plaquette lattice.

In [6]:
# Filter the plaquette lattice down to a single plaquette (12 qubits)
# so the AerSimulator run stays fast. The full lattice is used later
# in the large-scale hardware example.
gem_exp = GemExperiment(plaquette_lattice.filter([9]), backend=aer_backend)

# visualize the plaquette lattice after filtering
plaquette_lattice.filter([9]).draw_qubits()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/02357c6e-5c83-4ac0-811d-22602d9f33d5-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/a19d63ce-3572-4081-a008-c1332fbbe303-0.avif)

Bilang karagdagan sa mga qubit label at mga edge na nagsasaad kung aling mga qubit ang konektado, ang diagram ay naglalaman ng tatlong karagdagang piraso ng impormasyon na nauugnay sa GEM protocol:
- Ang bawat qubit ay may shading (gray) o walang shading. Ang mga shaded qubit ay "site" qubit na kumakatawan sa mga site ng Ising model, at ang mga unshaded qubit ay "bond" qubit na ginagamit upang magpagitna ng mga interaksyon sa pagitan ng mga site qubit.
- Ang bawat site qubit ay naka-label na (A) o (B), na nagsasaad ng isa sa dalawang tungkulin na maaaring gampanan ng isang site qubit sa GEM protocol (ang mga tungkulin ay ipaliliwanag sa ibang pagkakataon).
- Ang bawat edge ay may kulay gamit ang isa sa anim na kulay, kaya pinag-partition ang mga edge sa anim na grupo. Ang partition na ito ay tumutukoy kung paano maaaring i-parallelize ang mga two-qubit gate, gayundin ang iba't ibang mga scheduling pattern na malamang na magkaroon ng iba't ibang dami ng error sa isang maingay na quantum processor. Dahil ang mga edge sa isang grupo ay disjoint, ang isang layer ng two-qubit gates ay maaaring ilapat sa mga edge na iyon nang sabay-sabay. Sa katunayan, posible na hatiin ang anim na kulay sa tatlong grupo ng dalawang kulay na ang union ng bawat grupo ng dalawang kulay ay disjoint pa rin. Samakatuwid, tatlong layer lamang ng two-qubit gates ang kinakailangan upang ma-activate ang bawat edge. Mayroong 12 paraan upang i-partition ang anim na kulay sa ganitong paraan, at ang bawat partition na ito ay gumagawa ng ibang tatlong-layer na gate schedule.

Ngayong nakagawa ka na ng plaquette lattice, ang susunod na hakbang ay mag-initialize ng isang `GemExperiment` object, na nagpapasa ng parehong plaquette lattice at ng backend na balak mong patakbuhin ang eksperimento. Ang `GemExperiment` class ay namamahala sa aktwal na implementasyon ng GEM protocol, kasama ang paggawa ng mga circuit, pagsusumite ng mga job, at pagsusuri ng data. Ang sumusunod na code cell ay nag-i-initialize ng experiment class habang pinapaghigpit ang plaquette lattice sa isang plaquette lamang (12 qubit), na pinapanatiling maliit at mabilis ang simulation. Ang buong plaquette lattice ay gagamitin sa ibang pagkakataon kapag pinalawak sa tunay na hardware.

In [7]:
circuits = gem_exp.circuits()
print(f"Total number of circuits: {len(circuits)}")

Total number of circuits: 252


![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/02357c6e-5c83-4ac0-811d-22602d9f33d5-0.avif)

Ang isang GEM protocol circuit ay binubuo gamit ang sumusunod na mga hakbang:
1. Ihanda ang all-$|+\rangle$ state sa pamamagitan ng paglalapat ng Hadamard gate sa bawat qubit.
2. Maglapat ng $R_{ZZ}$ gate sa pagitan ng bawat pares ng konektadong qubit. Maaari itong gawin gamit ang tatlong layer ng mga gate. Ang bawat $R_{ZZ}$ gate ay kumikilos sa isang site qubit at isang bond qubit. Kung ang site qubit ay naka-label na (B), ang anggulo ay nakatakda sa $\frac{\pi}{2}$. Kung ang site qubit ay naka-label na (A), ang anggulo ay pinapayagang mag-iba, na gumagawa ng iba't ibang mga circuit. Bilang default, ang hanay ng mga anggulo ay nakatakda sa 21 pantay na espasyong mga punto sa pagitan ng $0$ at $\frac{\pi}{2}$, inclusive.
3. Sukatin ang bawat bond qubit sa Pauli $X$ basis. Dahil ang mga qubit ay sinusukat sa Pauli $Z$ basis, ito ay maaaring magawa sa pamamagitan ng paglalapat ng Hadamard gate bago sukatin ang qubit.

Tandaan na ang papel na binanggit sa panimula ng tutorial na ito ay gumagamit ng ibang convention para sa $R_{ZZ}$ angle, na naiiba sa convention na ginamit sa tutorial na ito ng factor na 2.

Sa hakbang tatlo, tanging ang mga bond qubit lamang ang sinusukat. Upang maintindihan kung anong estado ang nananatili sa mga site qubit, nakakatulong na isaalang-alang ang kaso na ang $R_{ZZ}$ angle na inilapat sa mga site qubit (A) sa hakbang dalawa ay katumbas ng $\frac{\pi}{2}$. Sa kasong ito, ang mga site qubit ay naiiwanan sa isang lubhang naka-entangle na estado na katulad ng GHZ state,

$$
\lvert \text{GHZ} \rangle = \lvert 00 \cdots 00 \rangle + \lvert 11 \cdots 11 \rangle.
$$

Dahil sa randomness sa mga measurement outcome, ang aktwal na estado ng mga site qubit ay maaaring ibang estado na may long-range order, halimbawa, $\lvert 00110 \rangle + \lvert 11001 \rangle$. Gayunpaman, ang GHZ state ay maaaring mabawi sa pamamagitan ng paglalapat ng decoding operation batay sa mga measurement outcome. Kapag ang $R_{ZZ}$ angle ay na-tune down mula sa $\frac{\pi}{2}$, ang long-range order ay maaari pa ring mabawi hanggang sa isang critical angle, na sa kawalan ng ingay ay humigit-kumulang $0.3 \pi$. Sa ibaba ng anggulo na ito, ang resultang estado ay hindi na nagpapakita ng long-range entanglement. Ang transisyon na ito sa pagitan ng presensya at kawalan ng long-range order ay ang Nishimori phase transition.

Sa paglalarawang nasa itaas, ang mga site qubit ay hindi sinukat, at ang decoding operation ay maaaring isagawa sa pamamagitan ng paglalapat ng mga quantum gate. Sa eksperimentong ipinatupad sa GEM suite, ang mga site qubit ay sinusukat, at ang decoding operation ay inilalapat sa isang classical post-processing step.

Sa paglalarawang nasa itaas, ang decoding operation ay maaaring isagawa sa pamamagitan ng paglalapat ng mga quantum gate sa mga site qubit upang mabawi ang quantum state. Gayunpaman, kung ang layunin ay agarang sukatin ang estado (halimbawa, para sa mga layunin ng characterization), maaari mong sukatin ang mga site qubit kasama ng mga bond qubit, at ilapat ang decoding operation sa isang classical post-processing step.

Bilang karagdagan sa pag-asa sa $R_{ZZ}$ angle sa hakbang dalawa, na bilang default ay sumasalakay sa 21 values, ang GEM protocol circuit ay umaasa rin sa scheduling pattern na ginamit upang ipatupad ang tatlong layer ng $R_{ZZ}$ gates. Tulad ng tinalakay dati, mayroong 12 gayong scheduling pattern. Samakatuwid, ang kabuuang bilang ng mga circuit sa eksperimento ay $21 \times 12 = 252$.

Ang mga circuit ng eksperimento ay maaaring i-generate gamit ang `circuits` method ng `GemExperiment` class.

In [8]:
# Restrict experiment to the first scheduling pattern
gem_exp.set_experiment_options(schedule_idx=0)

# There are less circuits now
circuits = gem_exp.circuits()
print(f"Total number of circuits: {len(circuits)}")

# Print the RZZ angles swept over
print(f"RZZ angles:\n{gem_exp.parameters()}")

Total number of circuits: 21
RZZ angles:
[0.         0.07853982 0.15707963 0.23561945 0.31415927 0.39269908
 0.4712389  0.54977871 0.62831853 0.70685835 0.78539816 0.86393798
 0.9424778  1.02101761 1.09955743 1.17809725 1.25663706 1.33517688
 1.41371669 1.49225651 1.57079633]


The following code cell draws a diagram of the circuit at index 5. To reduce the size of the diagram, the measurement gates at the end of the circuit are removed.

In [9]:
# Get the circuit at index 5
circuit = circuits[5]
# Remove the final measurements to ease visualization
circuit.remove_final_measurements()
# Draw the circuit
circuit.draw("mpl", fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/fd57d483-c70b-4ad5-b309-15750ad38bac-0.avif" alt="Output of the previous code cell" />

Para sa mga layunin ng tutorial na ito, sapat na na isaalang-alang ang isang scheduling pattern lamang. Ang sumusunod na code cell ay pinaghihigpit ang eksperimento sa unang scheduling pattern. Bilang resulta, ang eksperimento ay may 21 circuits lamang, isa para sa bawat $R_{ZZ}$ angle na sinasalakay.

In [10]:
# Demonstrate setting transpile options
gem_exp.set_transpile_options(
    optimization_level=1  # This is the default optimization level
)
pass_manager = generate_preset_pass_manager(
    backend=aer_backend,
    initial_layout=list(gem_exp.physical_qubits),
    **dict(gem_exp.transpile_options),
)
transpiled = pass_manager.run(circuit)
transpiled.draw("mpl", idle_wires=False, fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/e9b99d48-8d33-46b5-bff5-480ab1c1c1f2-0.avif" alt="Output of the previous code cell" />

### Step 3: Execute using Qiskit primitives

To execute the GEM protocol circuits on the hardware, call the `run` method of the `GemExperiment` object. You can specify the number of shots you want to sample from each circuit. The `run` method returns an [ExperimentData](https://qiskit-community.github.io/qiskit-experiments/stubs/qiskit_experiments.framework.ExperimentData.html) object which you should save to a variable. Note that the `run` method only submits jobs without waiting for them to finish, so it is a non-blocking call.

In [11]:
exp_data = gem_exp.run(shots=10_000)

Ang sumusunod na code cell ay gumaguhit ng diagram ng circuit sa index 5. Upang bawasan ang laki ng diagram, ang mga measurement gate sa dulo ng circuit ay inalis.

In [12]:
# The noiseless AerSimulator produces zero-variance UFloat objects in the
# analysis, which triggers a harmless warning from the `uncertainties`
# library. Suppress it so the output stays clean.
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore", message="Using UFloat objects with std_dev==0"
    )
    exp_data.block_for_results()
exp_data

ExperimentData(GemExperiment, 90bf2a90-f729-4c4e-a6da-664aecb11039, job_ids=['04a7c405-47fd-46ca-aa4b-aaf7e339cfbe'], metadata=<5 items>, figure_names=['two_point_correlation.svg', 'normalized_variance.svg', 'plaquette_ops.svg', 'bond_ops.svg'])

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/fd57d483-c70b-4ad5-b309-15750ad38bac-0.avif)

### Hakbang 2: I-optimize ang problema para sa quantum hardware execution
Ang pag-transpile ng mga quantum circuit para sa pagsasagawa sa hardware ay karaniwang nagsasangkot ng [ilang mga yugto](/guides/transpiler-stages). Kadalasan, ang mga yugtong may pinakamataas na computational overhead ay ang pagpili ng qubit layout, pag-route ng mga two-qubit gate upang umangkop sa qubit connectivity ng hardware, at pag-optimize ng circuit upang mabawasan ang gate count at depth nito. Sa GEM protocol, ang layout at routing stages ay hindi kinakailangan dahil ang hardware connectivity ay naisama na sa disenyo ng protocol. Ang mga circuit ay mayroon nang qubit layout, at ang mga two-qubit gate ay na-map na sa mga native connection. Higit pa rito, upang mapanatili ang istraktura ng circuit habang nag-iiba ang $R_{ZZ}$ angle, ang napaka-basic na circuit optimization lamang ang dapat isagawa.

Ang `GemExperiment` class ay transparent na nag-transpile ng mga circuit kapag isinasagawa ang eksperimento. Ang layout at routing stages ay na-override na bilang default upang walang gawin, at ang circuit optimization ay isinasagawa sa isang antas na nag-o-optimize lamang ng mga single-qubit gate. Gayunpaman, maaari mong i-override o magpasa ng karagdagang mga opsyon gamit ang `set_transpile_options` method. Para sa visualization, ang sumusunod na code cell ay manually nag-transpile ng circuit na ipinakita dati, at gumuhit ng transpiled circuit.

In [13]:
def magnetization_distribution(
    counts_dict: dict[str, int],
) -> dict[str, float]:
    """Compute magnetization distribution from counts dictionary."""
    # Construct dictionary from magnetization to count
    mag_dist = defaultdict(float)
    for bitstring, count in counts_dict.items():
        mag = bitstring.count("0") - bitstring.count("1")
        mag_dist[mag] += count
    # Normalize
    shots = sum(counts_dict.values())
    for mag in mag_dist:
        mag_dist[mag] /= shots
    return mag_dist


# Get counts dictionaries with and without decoding
data = exp_data.data()
# Get the last data point, which is at the angle for the GHZ state
raw_counts = data[-1]["counts"]
# Without decoding
site_indices = [
    i for i, q in enumerate(gem_exp.plaquettes.qubits()) if q.role == "Site"
]
site_raw_counts = defaultdict(int)
for key, val in raw_counts.items():
    site_str = "".join(key[-1 - i] for i in site_indices)
    site_raw_counts[site_str] += val
# With decoding
_, site_decoded_counts = gem_exp.plaquettes.decode_outcomes(
    raw_counts, return_counts=True
)

# Compute magnetization distribution
raw_magnetization = magnetization_distribution(site_raw_counts)
decoded_magnetization = magnetization_distribution(site_decoded_counts)

# Plot
plt.bar(*zip(*raw_magnetization.items()), label="raw")
plt.bar(*zip(*decoded_magnetization.items()), label="decoded", width=0.3)
plt.legend()
plt.xlabel("Magnetization")
plt.ylabel("Frequency")
plt.title("Magnetization distribution with and without decoding")

Text(0.5, 1.0, 'Magnetization distribution with and without decoding')

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/8ead3582-16df-4616-836c-bdce867ad6b8-1.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/e9b99d48-8d33-46b5-bff5-480ab1c1c1f2-0.avif)

### Hakbang 3: Isagawa gamit ang mga Qiskit primitive
Upang maisagawa ang mga GEM protocol circuit sa hardware, tawagan ang `run` method ng `GemExperiment` object. Maaari mong tukuyin ang bilang ng mga shot na nais mong i-sample mula sa bawat circuit. Ang `run` method ay nagbabalik ng isang [ExperimentData](https://qiskit-community.github.io/qiskit-experiments/stubs/qiskit_experiments.framework.ExperimentData.html) object na dapat mong i-save sa isang variable. Tandaan na ang `run` method ay nagsusumite lamang ng mga job nang hindi naghihintay na matapos ang mga ito, kaya ito ay isang non-blocking call.

In [14]:
exp_data.figure("two_point_correlation")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/4ecb25c8-e572-49af-a879-9943039db131-0.avif" alt="Output of the previous code cell" />

Upang maghintay para sa mga resulta, tawagan ang `block_for_results` method ng `ExperimentData` object. Ang tawag na ito ay magiging sanhi na ang interpreter ay magsususpende hanggang sa matapos ang mga job.

In [15]:
exp_data.figure("normalized_variance")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/2b351d68-3924-445a-94ef-047b16214e8a-0.avif" alt="Output of the previous code cell" />

## Large-scale hardware example

Having validated the protocol on a simulator, you can now scale up the experiment and run it on the real quantum hardware backend selected in the [Setup](#setup) section. This example uses two larger problem sizes:

- **Six plaquettes (~49 qubits)**: a mid-size run that already shows the rightward shift of the critical point under hardware noise.
- **The full plaquette lattice**: every plaquette the device's heavy-hex topology supports (for example, 18 plaquettes / 125 qubits on `ibm_torino` or 21 plaquettes / 144 qubits on `ibm_pittsburgh`), entangling qubits across the entire device with constant-depth circuits.

The single code cell below is self-contained: it builds the plaquette lattice from the backend's coupling map and runs both experiments, so this section can be executed after the [Setup](#setup) cells without first running the small-scale section.

In [ ]:
# -------------------------Step 1-------------------------
# Initialize the runtime service, pick a real quantum hardware backend,
# and build the plaquette lattice from its coupling map. This is repeated
# from the small-scale example so this cell can run standalone after the
# Setup section. The full plaquette lattice is the "large-scale" target;
# a six-plaquette subset (range(3, 9)) is also used to show an intermediate
# scaling step.
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
plaquette_lattice = PlaquetteLattice.from_coupling_map(backend.coupling_map)

# Build a GemExperiment for the full plaquette lattice and one for the
# six-plaquette subset, each restricted to a single scheduling pattern so
# the experiment has one circuit per RZZ angle (21 circuits total).
gem_exp_full = GemExperiment(plaquette_lattice, backend=backend)
gem_exp_full.set_experiment_options(schedule_idx=0)
gem_exp_6 = GemExperiment(
    plaquette_lattice.filter(range(3, 9)), backend=backend
)
gem_exp_6.set_experiment_options(schedule_idx=0)

circuits = gem_exp_full.circuits()
print(f"Total number of circuits (full lattice): {len(circuits)}")

# -------------------------Step 2-------------------------
# GemExperiment transpiles internally for the target backend: the layout
# and routing stages are overridden because the plaquette lattice already
# matches the hardware connectivity, and optimization is restricted so the
# RZZ angle structure is preserved. The code below manually transpiles one
# circuit from the six-plaquette experiment with the same settings this
# experiment will use, and draws it for inspection. (The full-lattice
# transpiled circuit has too many qubits to visualize cleanly, so the
# six-plaquette circuit is used here as a representative example.)
gem_exp_6.set_transpile_options(optimization_level=1)
circuits_6 = gem_exp_6.circuits()
pass_manager = generate_preset_pass_manager(
    backend=backend,
    initial_layout=list(gem_exp_6.physical_qubits),
    **dict(gem_exp_6.transpile_options),
)
transpiled = pass_manager.run(circuits_6[5])
display(transpiled.draw("mpl", idle_wires=False, fold=-1, scale=0.5))

# -------------------------Step 3-------------------------
# Run both problem sizes on real hardware:
#   1. Six plaquettes (~49 qubits) — an intermediate scale-up.
#   2. The full plaquette lattice — every plaquette the device supports.
exp_data_6 = gem_exp_6.run(shots=10_000, job_tags=["TUT_NPT"])
exp_data_full = gem_exp_full.run(shots=10_000, job_tags=["TUT_NPT"])
exp_data_6.block_for_results()
exp_data_full.block_for_results()

# -------------------------Step 4-------------------------
# Plot the normalized variance at each scale. The peak marks the critical
# point of the Nishimori transition; as the system grows, hardware noise
# shifts the peak rightward.
display(exp_data_6.figure("normalized_variance"))
exp_data_full.figure("normalized_variance")

Total number of circuits (full lattice): 21


<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/08581c09-a6a5-4a56-9fc4-abf22b063c6a-1.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/08581c09-a6a5-4a56-9fc4-abf22b063c6a-2.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/08581c09-a6a5-4a56-9fc4-abf22b063c6a-3.avif" alt="Output of the previous code cell" />

### Hakbang 4: I-post-process at ibalik ang resulta sa gustong classical format
Sa isang $R_{ZZ}$ angle na $\frac{\pi}{2}$, ang decoded state ay magiging GHZ state sa kawalan ng ingay. Ang long-range order ng GHZ state ay maaaring ivisualize sa pamamagitan ng pag-plot ng magnetization ng mga sinukatang bitstring. Ang magnetization $M$ ay tinukoy bilang kabuuan ng mga single-qubit Pauli $Z$ operator,
$$
M = \sum_{j=1}^N Z_j,
$$
kung saan ang $N$ ay ang bilang ng mga site qubit. Ang halaga nito para sa isang bitstring ay katumbas ng pagkakaiba sa pagitan ng bilang ng mga zero at bilang ng mga one. Ang pagsukat ng GHZ state ay nagbubunga ng all-zeros state o all-ones state na may pantay na probability, kaya ang magnetization ay magiging $+N$ kalahati ng oras at $-N$ ang kalahati ng oras. Sa presensya ng mga error dahil sa ingay, ang ibang mga value ay lilitaw din, ngunit kung ang ingay ay hindi masyadong malaki, ang distribusyon ay nakasentro pa rin malapit sa $+N$ at $-N$.

Para sa mga raw bitstring bago ang decoding, ang distribusyon ng magnetization ay magiging katumbas ng mga uniformly random bitstring, sa kawalan ng ingay.

Ang sumusunod na code cell ay nag-plot ng magnetization ng mga raw bitstring at decoded bitstring sa $R_{ZZ}$ angle na $\frac{\pi}{2}$.